In [1]:
import pandas as pd

df = pd.read_csv("../data/cleaned.csv")
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703.0,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286.0,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974.0,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817.0,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322.0,84,...,144,A,Minor,65,23,80,14,63,11,6


### Prediction Task

We aim to predict whether a song becomes a hit.

This is a **classification problem** because the output variable is binary:
- 1 → Hit
- 0 → Not a hit

Classification is more appropriate than regression because we are not interested in predicting the exact number of streams, but rather whether a song crosses a certain popularity threshold.

In [2]:
# Renaming some columns
streams_col = 'streams'
dance_col = 'danceability_%'
energy_col = 'energy_%'

# Target
df['hit'] = (df[streams_col] > df[streams_col].median()).astype(int)

# Feature Engineering
df['energy_dance'] = df[energy_col] * df[dance_col]
df['energy_squared'] = df[energy_col] ** 2


### Feature Engineering

We created two new features:

1. **energy_dance**
This feature combines energy and danceability to capture how suitable a song is for active listening environments like clubs.

2. **energy_squared**
This feature captures nonlinear effects of energy. Extremely high or low energy may influence popularity differently.

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Features / target
X = df.select_dtypes(include=['float64', 'int64']).drop('hit', axis=1)
y = df['hit']

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Models
lr = LogisticRegression(max_iter=1000)
dt = DecisionTreeClassifier()

In [4]:

lr_cv = cross_val_score(lr, X_train, y_train, cv=5, scoring='accuracy')
dt_cv = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')

print("LR CV Accuracy:", lr_cv.mean())
print("DT CV Accuracy:", dt_cv.mean())

LR CV Accuracy: 0.9938813857897827
DT CV Accuracy: 0.9954198473282443


In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Train
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)

# Predict
lr_pred = lr.predict(X_test)
dt_pred = dt.predict(X_test)

# Evaluation function
def evaluate(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "Confusion Matrix": confusion_matrix(y_true, y_pred)
    }

lr_metrics = evaluate(y_test, lr_pred)
dt_metrics = evaluate(y_test, dt_pred)

print("Logistic Regression:", lr_metrics)
print("Decision Tree:", dt_metrics)

Logistic Regression: {'Accuracy': 0.9878048780487805, 'Precision': 1.0, 'Recall': 0.9767441860465116, 'F1': 0.9882352941176471, 'Confusion Matrix': array([[78,  0],
       [ 2, 84]])}
Decision Tree: {'Accuracy': 0.9939024390243902, 'Precision': 1.0, 'Recall': 0.9883720930232558, 'F1': 0.9941520467836257, 'Confusion Matrix': array([[78,  0],
       [ 1, 85]])}


In [6]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy": [lr_metrics["Accuracy"], dt_metrics["Accuracy"]],
    "Precision": [lr_metrics["Precision"], dt_metrics["Precision"]],
    "Recall": [lr_metrics["Recall"], dt_metrics["Recall"]],
    "F1 Score": [lr_metrics["F1"], dt_metrics["F1"]]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.987805,1.0,0.976744,0.988235
1,Decision Tree,0.993902,1.0,0.988372,0.994152


### Conclusion

The Decision Tree model performed better than Logistic Regression based on the evaluation metrics, particularly the F1-score. While Logistic Regression is simpler and assumes a linear relationship, the Decision Tree can capture complex nonlinear patterns in the data.

In practical terms, prediction errors have important consequences. A false positive means predicting a song will be a hit when it is not, which could lead to wasted promotional resources. A false negative means failing to identify a potential hit, resulting in missed opportunities. Therefore, improving model performance helps music producers and platforms make better strategic decisions.

In [7]:

import joblib

# Save errors
errors = X_test.copy()
errors['actual'] = y_test
errors['lr_pred'] = lr_pred
errors['dt_pred'] = dt_pred
errors.to_csv("../data/prediction_errors.csv", index=False)

# Save best model
best_model = dt if dt_metrics["F1"] > lr_metrics["F1"] else lr
joblib.dump(best_model, "../models/supervised_best.pkl")
comparison.to_csv("../reports/task2_results.csv", index=False)